# 06 · Verificación por alias y exportación

La evaluación coteja las fechas, las huellas de datos y el ROC-AUC de las versiones a las que apuntan los alias. Si la extracción difiere, informa qué huella no coincide. La exportación es opcional.

In [7]:
VERIFICAR_ALIAS = True
EXPORTAR_CORRIDAS = False

In [2]:
import json
import os
import sys
from hashlib import sha256
from pathlib import Path
from time import perf_counter

import mlflow
import mlflow.sklearn
import pandas as pd

from mlflow import MlflowClient
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.svm import SVC

directorio = Path.cwd().resolve()
raiz = next(
    (
        ruta for ruta in [directorio, *directorio.parents]
        if (ruta / "src" / "pipeline.py").is_file()
    ),
    None,
)

if raiz is None:
    raise FileNotFoundError("Abre el notebook desde el repositorio pset3.")

sys.path.insert(0, str(raiz))

from src.datos import cargar_obt
from src.features import construir_features
from src.pipeline import crear_pipeline

SEMILLA = 42
HORIZONTE = 5
CUANTIL_TARGET = 0.80
inicio_prueba = pd.Timestamp("2023-07-11")
capital_inicial = 10_000.0

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_registry_uri("http://127.0.0.1:5000")
cliente = MlflowClient()
NOMBRE_EXPERIMENTO = "Sistema de alerta temprana de volatilidad para acciones"
NOMBRE_REGISTRO = "alerta-volatilidad-aapl-modelo"
ruta_revision = raiz / "docs" / "revision_runs.json"

def guardar_ids(contenido):
    temporal = ruta_revision.with_suffix(".tmp")
    ruta_revision.parent.mkdir(exist_ok=True)
    temporal.write_text(json.dumps(contenido, indent=2), encoding="utf-8")
    temporal.replace(ruta_revision)

revision = (
    json.loads(ruta_revision.read_text(encoding="utf-8"))
    if ruta_revision.is_file() else {}
)

c:\Users\usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [3]:
df = cargar_obt()
snapshot = df.sort_values(["TRADE_DATE", "SYMBOL"])

datos_corrida = {
    "tabla_origen": "PSET2_DB.OBT.AAPL_DAILY_OBT",
    "fecha_corte_snapshot": os.environ["DATA_CUTOFF"],
    "sha256_snapshot_obt": sha256(
        snapshot.to_csv(index=False, lineterminator="\n").encode("utf-8")
    ).hexdigest(),
    "filas_snapshot": len(snapshot),
}

print("Dimensiones de la OBT:", df.shape)
print(datos_corrida)

Dimensiones de la OBT: (4087, 19)
{'tabla_origen': 'PSET2_DB.OBT.AAPL_DAILY_OBT', 'fecha_corte_snapshot': '2026-09-30', 'sha256_snapshot_obt': '673213bc5b0fe2ef795cd2492ece28723d4b3d92656aadfb5cadb71ed13dbafb', 'filas_snapshot': 4087}


In [4]:
features = construir_features(df)

vol_futura_5 = (
    features["RET_LAG_0"]
    .rolling(HORIZONTE, min_periods=HORIZONTE)
    .std(ddof=1)
    .shift(-HORIZONTE)
    .rename("VOL_FUTURA_5")
)

fin_ventana_target = features.index.to_series().shift(-HORIZONTE)

datos_modelado = (
    features.iloc[60:]
    .join(vol_futura_5)
    .dropna(subset=["VOL_FUTURA_5"])
)

entrenamiento = datos_modelado.loc[
    fin_ventana_target.reindex(datos_modelado.index) < inicio_prueba
].copy()

fechas_holdout = datos_modelado.index[datos_modelado.index >= inicio_prueba]

X_desarrollo = entrenamiento.drop(columns="VOL_FUTURA_5")
vol_desarrollo = entrenamiento["VOL_FUTURA_5"]
columnas_numericas = X_desarrollo.columns.drop("MES").tolist()
cv_temporal = TimeSeriesSplit(n_splits=5, gap=HORIZONTE)

hash_desarrollo = sha256(
    entrenamiento.to_csv(index=True, lineterminator="\n").encode("utf-8")
).hexdigest()

# Retorno de t a t+1, expresado como proporción y alineado con t.
retorno_siguiente = features["RET_LAG_0"].shift(-1).div(100)

assert fin_ventana_target.loc[X_desarrollo.index].max() < fechas_holdout.min()

print("Desarrollo:", X_desarrollo.shape)
print("Fechas reservadas para prueba:", len(fechas_holdout))

Desarrollo: (3212, 18)
Fechas reservadas para prueba: 805


In [5]:
if VERIFICAR_ALIAS:
    from math import isclose

    X_prueba = datos_modelado.loc[fechas_holdout, X_desarrollo.columns]
    vol_prueba = datos_modelado.loc[fechas_holdout, "VOL_FUTURA_5"]
    verificacion = []

    for alias in ("champion", "challenger"):
        uri = f"models:/{NOMBRE_REGISTRO}@{alias}"
        version = cliente.get_model_version_by_alias(NOMBRE_REGISTRO, alias)
        corrida = cliente.get_run(version.tags.get("run_id_final") or version.run_id)
        params = corrida.data.params
        auc_registrado = corrida.data.metrics["roc_auc_test"]
        checks = {
            "OBT": params.get("sha256_snapshot_obt") == datos_corrida["sha256_snapshot_obt"],
            "desarrollo": params.get("sha256_dataset_desarrollo") == hash_desarrollo,
            "filas hold-out": str(len(X_prueba)) == params.get("filas_holdout"),
            "inicio hold-out": X_prueba.index[0].date().isoformat() == params.get("inicio_holdout"),
            "fin hold-out": X_prueba.index[-1].date().isoformat() == params.get("fin_holdout"),
        }
        diferencias = [nombre for nombre, coincide in checks.items() if not coincide]
        modelo = mlflow.sklearn.load_model(uri)
        auc_actual = None
        if diferencias:
            estado = "Datos distintos: " + ", ".join(diferencias)
        else:
            y = vol_prueba.gt(float(params["p80_final"])).astype(int)
            auc_actual = roc_auc_score(y, modelo.predict_proba(X_prueba)[:, 1])
            estado = (
                "Reproducido"
                if isclose(auc_actual, auc_registrado, rel_tol=0, abs_tol=1e-12)
                else "ROC-AUC distinto"
            )

        verificacion.append({
            "alias": alias, "version": version.version, "run_id": corrida.info.run_id,
            "auc_registrado": auc_registrado, "auc_recalculado": auc_actual, "estado": estado,
        })

    display(pd.DataFrame(verificacion).round(12))

,alias,version,run_id,auc_registrado,auc_recalculado,estado
0,champion,3,cdfde8f825a141ef9fd839389c099b4f,0.635553,0.635553,Reproducido
1,challenger,4,e51139f81b334a529ac08e14cc4b71b6,0.643915,0.643915,Reproducido


In [8]:
if EXPORTAR_CORRIDAS:
    padres_tuning = revision.get("tuning", {})
    assert len(padres_tuning) == 3, (
        "Se esperaban tres IDs padre en docs/revision_runs.json."
    )

    nombres_experimentos = [
        NOMBRE_EXPERIMENTO,
        "alerta-volatilidad-aapl-ablacion",
        "alerta-volatilidad-aapl-modelos-fundacionales",
    ]
    experimentos = [
        cliente.get_experiment_by_name(nombre)
        for nombre in nombres_experimentos
    ]
    ids = [exp.experiment_id for exp in experimentos if exp is not None]
    corridas_pset = mlflow.search_runs(experiment_ids=ids)

    ids_busqueda_hijas = sorted(set(ids) | {"0"})
    hijas_tuning = []

    for modelo, run_id_padre in padres_tuning.items():
        hijas = mlflow.search_runs(
            experiment_ids=ids_busqueda_hijas,
            filter_string=(
                f"tags.`mlflow.parentRunId` = '{run_id_padre}' "
                "AND attributes.status = 'FINISHED'"
            ),
        )
        cantidad = hijas["run_id"].nunique()
        assert cantidad == 30, f"{modelo}: se encontraron {cantidad} de 30 hijas."
        print(f"{modelo}: {cantidad}/30 corridas hijas verificadas")
        hijas_tuning.append(hijas)

    corridas = pd.concat(
        [corridas_pset, *hijas_tuning], ignore_index=True
    ).drop_duplicates(subset="run_id")

    ruta_csv = raiz / "docs" / "corridas.csv"
    ruta_csv.parent.mkdir(exist_ok=True)
    ruta_temporal = ruta_csv.with_suffix(".tmp")
    corridas.to_csv(ruta_temporal, index=False)
    ruta_temporal.replace(ruta_csv)

    print(f"{len(corridas)} corridas exportadas a {ruta_csv}")

RandomForestClassifier: 30/30 corridas hijas verificadas
LogisticRegression: 30/30 corridas hijas verificadas
SVC: 30/30 corridas hijas verificadas
212 corridas exportadas a C:\USFQ\Maestría\Ciencia_Datos\pset3\docs\corridas.csv
